## 知识蒸馏 — 让 MLP 学到 ResNet 的「暗知识」

### 核心思想

让一个大的"教师"模型（ResNet，**~99.5%** 准确率）教一个小的"学生"模型（MLP，**~537K** 参数）。

### 传统训练 vs 蒸馏训练

```
传统训练:
  图片 → MLP学生 → [0.01, 0.02, 0.90, ...] → 对比真实标签"2" → 只知道"这是2"

蒸馏训练:
  图片 → ResNet教师 → [0.00, 0.05, 0.85, 0.03, 0.07, ...]
                            ↓ 软标签：不仅告诉学生"这是2"，
                            ↓ 还告诉学生"有点像3"，"有点像5"
  图片 → MLP学生   → [0.02, 0.08, 0.75, 0.06, 0.09, ...]
                            ↓ 学生同时学习: ①真实标签 ②教师的软标签
```

### 蒸馏损失公式

```
Loss = α × CrossEntropy(学生输出, 真实标签)                    ← 硬标签（正确答案）
     + (1-α) × T² × KL(Softmax(学生/T) || Softmax(教师/T))    ← 软标签（类别关系）

其中:
  α: 硬标签权重（通常 0.5~0.9）
  T: 温度（通常 3~10），越高分布越"平滑"
  T²: 缩放因子，补偿高温下梯度缩小
```

> 教师: ResNet (99.5%)，学生: MLP (537K params)。观察蒸馏后 MLP 能否超越独立训练的 MLP (98.6%)。

## 1. 导入依赖与数据加载

需要同时准备两种输入格式：flat（学生 MLP 用）和 image（教师 ResNet 用）。

In [ ]:
# [1.1 导入依赖]
# ResNet(教师) → MLP(学生) | 软标签传递暗知识 | Loss = α×CE + (1-α)×T²×KL
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
import warnings
warnings.filterwarnings('ignore')

# ==== 中文字体配置 ====
# matplotlib 默认不支持中文，需手动指定支持中文的字体
# 优先级：微软雅黑 > 黑体 > DejaVu Sans（英文回退）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 防止负号显示为方块

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')

In [ ]:
# [1.2 加载 MNIST 数据 — 双格式并行]
# flat:  MLP 学生用展平 (N, 784)
# image: ResNet 教师用图像格式 (N, 1, 28, 28)
mnist = fetch_openml(
    name='mnist_784', version=1, as_frame=False,
    cache=True, data_home='../data'
)
X = mnist.data.astype(np.uint8)
y = mnist.target.astype(np.uint8)
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

# MLP 用展平 (N, 784)
X_train_flat = torch.tensor(X_train, dtype=torch.float32) / 255.0
X_test_flat  = torch.tensor(X_test,  dtype=torch.float32) / 255.0
# ResNet 用图像格式 (N, 1, 28, 28)
X_train_img  = torch.tensor(X_train.reshape(-1, 28, 28), dtype=torch.float32).unsqueeze(1) / 255.0
X_test_img   = torch.tensor(X_test.reshape(-1, 28, 28),  dtype=torch.float32).unsqueeze(1) / 255.0
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# ======== 超参数集中修改区 ========
BATCH_SIZE = 128
# =================================

# MLP 用 DataLoader
train_loader_flat = DataLoader(TensorDataset(X_train_flat, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader_flat  = DataLoader(TensorDataset(X_test_flat,  y_test_t),  batch_size=BATCH_SIZE, shuffle=False)
# ResNet 用 DataLoader
train_loader_img = DataLoader(TensorDataset(X_train_img, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader_img  = DataLoader(TensorDataset(X_test_img,  y_test_t),  batch_size=BATCH_SIZE, shuffle=False)

print(f'训练集: {X_train.shape[0]}, 测试集: {X_test.shape[0]}')

## 2. 定义教师（ResNet）和学生（MLP）模型

结构分别与 `06_resnet_mnist.ipynb` 和 `02_mlp_training.ipynb` 完全一致。
模型结构必须与训练时保持一致，否则权重加载会失败。

In [ ]:
# [2. 定义模型]
#  教师模型：ResNet-14（与 06_resnet_mnist.ipynb 完全一致）
class BasicBlock(nn.Module):
    """
    ResNet 基础残差块

    主路: Conv → BN → ReLU → Conv → BN
    短路: 恒等映射 (或 1×1 卷积对齐维度)
    输出: ReLU(主路 + 短路)
    """
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        """残差块前向: F(x) + x → ReLU"""
        # 1. 短路分支（恒等映射或 1×1 投影）
        identity = self.shortcut(x)
        # 2. 主路: Conv → BN → ReLU → Conv → BN
        out = nn.ReLU()(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # 3. 残差连接: F(x) + x → ReLU
        out += identity
        return nn.ReLU()(out)


class ResNet(nn.Module):
    """轻量 ResNet，为 MNIST 设计"""
    def __init__(self, block, num_blocks, num_classes=10,
                 in_planes=32, channel_list=None):
        super(ResNet, self).__init__()
        if channel_list is None:
            channel_list = [32, 64, 128]
        self.in_channels = in_planes
        self.num_layers = len(channel_list)
        # ==== 初始卷积层 (B,1,28,28) → (B,32,28,28) ====
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, in_planes, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(in_planes), nn.ReLU()
        )
        # ==== 三层残差 Layer（layer1不被下采样） ====
        for i, (ch, nb) in enumerate(zip(channel_list, num_blocks)):
            stride = 1 if i == 0 else 2
            layer = self._make_layer(block, ch, nb, stride=stride)
            setattr(self, f'layer{i + 1}', layer)
        # ==== 全局平均池化 + 分类头 ====
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channel_list[-1], num_classes)
        # ==== Kaiming 初始化 ====
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        """构建一个残差层: 第一个block负责通道/尺寸变换"""
        layers = [block(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(block(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        """前向: conv1 → layer1~3 → avgpool → fc"""
        # 1. 初始卷积: (B,1,28,28) → (B,32,28,28)
        x = self.conv1(x)
        # 2. 三层残差: layer1(28×28) → layer2(14×14) → layer3(7×7)
        for i in range(1, self.num_layers + 1):
            x = getattr(self, f'layer{i}')(x)
        # 3. 全局平均池化: (B,128,7,7) → (B,128,1,1)
        x = self.avgpool(x)
        # 4. 展平 + 分类头: (B,128) → (B,10)
        return self.fc(torch.flatten(x, 1))


#  学生模型：MLP（与 02_mlp_training.ipynb 完全一致）
class MLP(nn.Module):
    """
    多层感知机 — 学生模型

    结构：Linear(784→512) → BN → ReLU → Dropout
         → Linear(512→256) → BN → ReLU → Dropout
         → Linear(256→10)
    """
    def __init__(self, input_dim=784, hidden_dims=None, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        if hidden_dims is None:
            hidden_dims = [512, 256]
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        """前向: (B,784) → Sequential(Linear→BN→ReLU→Dropout)×2 → Linear → (B,10) logits"""
        return self.net(x)


print('教师模型: ResNet-14')
print('学生模型: MLP [512, 256]')

## 3. 加载预训练教师模型

教师参数**冻结**，蒸馏过程中不更新。如果还没训练 ResNet，先运行 `06_resnet_mnist.ipynb`。

In [ ]:
# [3. 加载教师模型并验证]
# 冻结所有参数 | .eval() + no_grad 确保只推理不训练 | 验证教师准确率
teacher = ResNet(BasicBlock, num_blocks=[2, 2, 2], num_classes=10).to(device)
state = torch.load('../models/resnet/best.pth', map_location=device)
teacher.load_state_dict(state)
teacher.eval()

# 冻结教师所有参数
for param in teacher.parameters():
    param.requires_grad = False

# 验证教师准确率
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader_img:
        images, labels = images.to(device), labels.to(device)
        outputs = teacher(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

teacher_acc = correct / total
teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'教师 (ResNet) 测试准确率: {teacher_acc:.4f} ({correct}/{total})')
print(f'教师参数量: {teacher_params:,}')

## 4. 理解温度与蒸馏损失

### 温度 T 的作用

```
T=1:  softmax → [0.001, 0.002, 0.990, ...]   ← 输出很"尖锐"，接近 one-hot
T=5:  softmax → [0.02,  0.05,  0.80,  ...]   ← 输出变"平滑"
T=10: softmax → [0.05,  0.08,  0.50,  ...]   ← 更平滑，"暗知识"更明显
```

T 越高，教师输出的概率分布越"平滑"，类别间的相似性信息（暗知识）越容易被学生学到。

### 为什么需要 T² 缩放？

高温下 softmax 的梯度近似缩小 T² 倍。乘回 T² 保证软标签损失和硬标签损失在同一个量级。

In [ ]:
# [4. 蒸馏损失函数 + 温度可视化]
def distillation_loss(student_logits, teacher_logits, labels, T, alpha):
    """
    知识蒸馏总损失

    Loss = α × CrossEntropy(student, labels)        ← 硬标签
         + (1-α) × T² × KL(student/T || teacher/T)  ← 软标签

    Args:
        student_logits: 学生模型原始输出 (未经过 softmax)
        teacher_logits: 教师模型原始输出
        labels: 真实标签
        T: 温度参数，越高软化越强（推荐 3~10）
        alpha: 硬标签权重 [0, 1]（推荐 0.5~0.9）
    """
    # 硬标签损失：标准交叉熵
    hard_loss = F.cross_entropy(student_logits, labels)

    # 软标签损失：KL 散度
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean')

    # T² 缩放 + 加权组合
    return alpha * hard_loss + (1 - alpha) * (T * T) * soft_loss


def visualize_temperature():
    """
    可视化温度对 softmax 分布的影响

    T 越高 → 输出越平滑 → 暗知识越暴露
    """
    logits = np.array([5.0, 1.0, 0.3, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001])
    logits_tensor = torch.tensor(logits, dtype=torch.float32)

    temperatures = [1.0, 2.0, 5.0, 10.0]
    fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 3.5))
    for ax, T in zip(axes, temperatures):
        probs = F.softmax(logits_tensor / T, dim=0).numpy()
        colors = plt.cm.viridis(np.linspace(0, 1, 10))
        ax.bar(range(10), probs, color=colors)
        ax.set_title(f'T = {T}')
        ax.set_xlabel('类别')
        ax.set_ylabel('概率')
        ax.set_ylim(0, 1.05)
        top3 = np.argsort(probs)[-3:][::-1]
        for i, idx in enumerate(top3):
            ax.text(idx, probs[idx] + 0.03, f'{probs[idx]:.2f}', ha='center', fontsize=8)
    plt.suptitle('温度对 Softmax 分布的影响：T 越高 → 分布越平滑 → "暗知识"越暴露',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_temperature()

## 5. 蒸馏训练函数

同步迭代两个 DataLoader：flat（学生 MLP）+ img（教师 ResNet）。教师始终 eval 模式且不计算梯度。

In [ ]:
# [5. 蒸馏训练函数]
def train_one_epoch_distill(student, teacher, loader_flat, loader_img, optimizer, T, alpha, device):
    """
    蒸馏训练一个 epoch

    同步迭代 flat(学生MLP) 和 img(教师ResNet) 两个 DataLoader
    教师始终 eval 模式，不计算梯度，只提供软标签

    Returns:
        (avg_loss, accuracy)
    """
    student.train()
    teacher.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for (flat_img, labels), (img, _) in zip(loader_flat, loader_img):
        flat_img = flat_img.to(device)
        img = img.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # ① 学生前向（展平输入）
        student_logits = student(flat_img)

        # ② 教师前向（图像格式，无梯度，只提供软标签）
        with torch.no_grad():
            teacher_logits = teacher(img)

        # ③ 蒸馏损失 = α×硬标签(CE) + (1-α)×T²×软标签(KL)
        loss = distillation_loss(student_logits, teacher_logits, labels, T, alpha)

        # ④ 反向传播 + 参数更新
        loss.backward()
        optimizer.step()

        # 累积统计
        total_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(student_logits, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, device):
    """评估模型，返回 (avg_loss, accuracy)"""
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            # 前向 + CE损失（不反向传播）
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            # 累加统计
            total_loss += loss.item() * labels.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

## 6. 启动蒸馏训练

蒸馏超参数 T（温度）和 α（硬标签权重）是核心。T=4.0, α=0.7 是常用起始点。

In [ ]:
# [6. 蒸馏训练]

# ======== 超参数集中修改区 ========
T = 4.0           # 温度：越高教师输出越"软化"，推荐 3~10
alpha = 0.7       # 硬标签权重：0.7 = 70% 正确答案 + 30% 模仿教师
EPOCHS = 50
LR = 0.001
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 5
# =================================

student = MLP(input_dim=784, hidden_dims=[512, 256], num_classes=10, dropout=0.2).to(device)
student_params = sum(p.numel() for p in student.parameters())
print(f'学生参数量: {student_params:,}')
print(f'温度 T={T}, 硬标签权重 α={alpha}')
print(f'压缩比: {student_params / teacher_params:.1%} (学生/教师)')

optimizer_distill = optim.Adam(student.parameters(), lr=LR)
scheduler_distill = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_distill, mode='min', factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE, verbose=True
)

history_distill = {'test_loss': [], 'test_acc': []}
best_test_acc_distill = 0.0

print('\n开始蒸馏训练...')
for epoch in range(1, EPOCHS + 1):
    # ==== 1. 蒸馏训练一个 epoch（学生学教师的软标签） ====
    train_loss, train_acc = train_one_epoch_distill(
        student, teacher, train_loader_flat, train_loader_img,
        optimizer_distill, T, alpha, device
    )
    # ==== 2. 测试集评估 ====
    test_loss, test_acc = evaluate(student, test_loader_flat, device)
    # ==== 3. ReduceLROnPlateau 根据验证损失自动衰减学习率 ====
    scheduler_distill.step(test_loss)

    # ==== 4. 记录训练历史 ====
    history_distill['test_loss'].append(test_loss)
    history_distill['test_acc'].append(test_acc)

    # ==== 5. 保存最优模型 ====
    if test_acc > best_test_acc_distill:
        best_test_acc_distill = test_acc
        torch.save(student.state_dict(), '../models/distill/best.pth')

    # ==== 6. 每 5 个 epoch 或第 1 个 epoch 打印进度 ====
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:3d}/{EPOCHS}]  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}  loss={test_loss:.4f}')

print(f'\n蒸馏完成！学生最佳测试准确率: {best_test_acc_distill:.4f}')

## 7. 对比实验：独立训练 MLP（无蒸馏）

用完全相同的 MLP 架构和超参数，仅使用标准交叉熵损失训练，作为对照基线。
这样才能公平地判断蒸馏是否有效。

In [ ]:
# [7. 对比基线 — 独立训练 MLP]
# 相同架构 + 相同超参数 + 纯 CE 损失 → 作为蒸馏的对照基线
student_baseline = MLP(input_dim=784, hidden_dims=[512, 256], num_classes=10, dropout=0.2).to(device)

optimizer_baseline = optim.Adam(student_baseline.parameters(), lr=LR)
scheduler_baseline = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_baseline, mode='min', factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE, verbose=False
)
criterion_ce = nn.CrossEntropyLoss()

history_baseline = {'test_loss': [], 'test_acc': []}
best_test_acc_baseline = 0.0

print('训练基线 MLP（无蒸馏）...')
for epoch in range(1, EPOCHS + 1):
    # ==== 1. 标准训练一个 epoch（仅 CE 损失，无蒸馏） ====
    student_baseline.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader_flat:
        images, labels = images.to(device), labels.to(device)
        # ① 梯度清零
        optimizer_baseline.zero_grad()
        # ② 前向传播
        outputs = student_baseline(images)
        # ③ CE 损失
        loss = criterion_ce(outputs, labels)
        # ④ 反向传播
        loss.backward()
        # ⑤ 参数更新
        optimizer_baseline.step()
        # 累积统计
        total_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    # ==== 2. 测试集评估 ====
    test_loss, test_acc = evaluate(student_baseline, test_loader_flat, device)
    # ==== 3. ReduceLROnPlateau 衰减学习率 ====
    scheduler_baseline.step(test_loss)

    # ==== 4. 记录历史 ====
    history_baseline['test_loss'].append(test_loss)
    history_baseline['test_acc'].append(test_acc)

    # ==== 5. 保存最优 ====
    if test_acc > best_test_acc_baseline:
        best_test_acc_baseline = test_acc

    # ==== 6. 打印进度 ====
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch [{epoch:3d}/{EPOCHS}]  test_acc={test_acc:.4f}  loss={test_loss:.4f}')

print(f'\n基线 MLP 完成！最佳测试准确率: {best_test_acc_baseline:.4f}')

## 8. 结果对比

**核心问题**：蒸馏后的 MLP 能否超越独立训练的 MLP？提升多少？

In [ ]:
# [8. 结果对比]
# Teacher vs MLP(baseline) vs MLP(distilled) 三栏对比
improvement = best_test_acc_distill - best_test_acc_baseline

print('=' * 60)
print(f'{"Model":<22} {"Test Acc":<12} {"Params":<12}')
print('=' * 60)
print(f'{"Teacher (ResNet)":<22} {teacher_acc:<12.4f} {teacher_params:<12,}')
print(f'{"MLP (baseline)":<22} {best_test_acc_baseline:<12.4f} {student_params:<12,}')
print(f'{"MLP (distilled)":<22} {best_test_acc_distill:<12.4f} {student_params:<12,}')
print('-' * 60)

if improvement > 0:
    print(f'蒸馏提升: +{improvement:.4f} ({improvement*100:.2f}%)')
    gap_to_teacher_before = teacher_acc - best_test_acc_baseline
    gap_to_teacher_after  = teacher_acc - best_test_acc_distill
    recovered = (gap_to_teacher_before - gap_to_teacher_after) / max(gap_to_teacher_before, 1e-8) * 100
    print(f'师生差距缩小: {recovered:.1f}%')
else:
    print(f'无提升: {improvement:.4f}')

print('=' * 60)

In [ ]:
# [9. 可视化对比]
# 蒸馏 vs 基线 Test Accuracy / Loss 曲线 | 绿色虚线 = 教师准确率
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：Accuracy
axes[0].plot(history_baseline['test_acc'], label='MLP (baseline)', linewidth=1.5)
axes[0].plot(history_distill['test_acc'], label='MLP (distilled)', linewidth=1.5)
axes[0].axhline(y=teacher_acc, color='green', linestyle='--', alpha=0.5,
                label=f'Teacher: {teacher_acc:.4f}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('蒸馏 vs 基线: 测试准确率')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：Loss
axes[1].plot(history_baseline['test_loss'], label='MLP (baseline)', linewidth=1.5)
axes[1].plot(history_distill['test_loss'], label='MLP (distilled)', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Loss')
axes[1].set_title('蒸馏 vs 基线: 测试损失')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 进一步学习

- 改变 T（1~20）和 α（0.1~0.9），找最优组合
- 尝试用 CNN 做教师、更小的 MLP 做学生
- 对比不同温度下学生模型的混淆矩阵变化